# MetaGPT
不同于 AutoGen 强调的“自由对话与代码执行”，MetaGPT 的核心哲学是将软件公司的标准化操作流程（Standard Operating Procedures, SOP）注入到 Multi-Agent 系统中。

## MetaGPT 的 SOP 驱动与结构化多 Agent 协作
1. 理解 MetaGPT 的核心理念：掌握 SOP（标准化作业程序）如何解决 Multi-Agent 对话中的“噪声膨胀”与“效率低下”问题。
2. 解构 MetaGPT 的两大基石：深入理解 Role（角色） 封装与基于 Publish-Subscribe（发布-订阅） 的消息路由机制。
3. 工程实战：基于 Python/LangGraph 原生模拟 MetaGPT 架构，实现“软件公司 SOP”下的需求分析与架构设计协作链。

#### 为什么 Multi-Agent 需要 SOP？
在传统的无约束 Multi-Agent 对话（如自由群聊）中，Agent 之间往往会产生大量重复、泛化甚至偏离主题的废话，导致 Token 消耗极快且任务交付质量不可控。

MetaGPT 提出了解决方案——引入 SOP（标准化作业程序）：

[自然语言需求] -> (ProductManager) -> [PRD 结构化文档]
                     ↓ (发布到共享消息总线)
                 (Architect) -> [System Architecture 架构设计]
                     ↓ (发布到共享消息总线)
                 (Engineer)  -> [可运行的代码实现]

1. 结构化输出约束（Structured Communication）：
    * 角色之间不传递毫无格式的自由文本，而是必须输出固定的结构化产物（如 Markdown 格式的需求文档 PRD、UML 架构图、标准 API 定义）。

2. 基于订阅的消息路由（Publish-Subscribe Pattern）：
    * 每个 Agent 只需要订阅与其职责相关的特定消息类型。例如，`Architect`（架构师）只关注 `ProductManager` 输出的 PRD，而不必监听其他无关角色的闲聊消息，极大节省了 Context 空间。

#### 原生模拟 MetaGPT 的“PRD 需求分析 $\rightarrow$ 系统架构设计”SOP 协作
为了让你看清底层原理，我们将不依赖 MetaGPT 的高层 SDK，而是使用 Pydantic + LangChain 手写模拟 MetaGPT 的结构化 SOP 协作流程。


In [ ]:
from typing import List, Annotated, TypedDict
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# ================= 1. 定义 SOP 的结构化产物 (Data Standard) =================

class PRDDocument(BaseModel):
    """产品经理输出的标准 PRD 文档"""
    project_name: str = Field(description="项目名称")
    core_goals: List[str] = Field(description="核心功能目标列表")
    user_stories: List[str] = Field(description="典型用户故事")

class ArchitectureDesign(BaseModel):
    """架构师输出的标准架构设计文档"""
    tech_stack: List[str] = Field(description="推荐的技术栈列表")
    data_flow: str = Field(description="核心数据流转逻辑描述")
    api_endpoints: List[str] = Field(description="预期的核心 API 接口清单")

# ================= 2. 定义全局通信 State =================

class MetaGPTState(TypedDict):
    user_requirement: str
    prd_output: PRDDocument | None
    arch_output: ArchitectureDesign | None
    messages: Annotated[list[BaseMessage], add_messages]

llm = ChatOpenAI(model="gpt-4o", temperature=0)

# ================= 3. 定义 MetaGPT 角色节点 (Role Nodes) =================

def product_manager_role(state: MetaGPTState):
    """【PM 角色】：执行 SOP 步骤 1 - 将自然语言需求转化为结构化 PRD"""
    pm_llm = llm.with_structured_output(PRDDocument)

    prompt = f"请作为一名资深产品经理(PM)，将用户的原始需求标准化转换为标准的 PRD 文档。需求：{state['user_requirement']}"
    prd_res = pm_llm.invoke(prompt)

    # 构造输出消息存入总线
    msg_content = f"【PM 产出 PRD】项目: {prd_res.project_name}\n目标: {prd_res.core_goals}"

    return {
        "prd_output": prd_res,
        "messages": [HumanMessage(content=msg_content, name="ProductManager")]
    }

def architect_role(state: MetaGPTState):
    """【架构师角色】：执行 SOP 步骤 2 - 订阅 PRD 并输出结构化架构设计"""
    arch_llm = llm.with_structured_output(ArchitectureDesign)

    # 强行依赖 upstream 的结构化 PRD 输出
    prd: PRDDocument = state["prd_output"]
    prompt = (
        f"请作为一名首席架构师(Architect)，严格根据以下 PRD 文档进行系统架构设计：\n"
        f"项目名：{prd.project_name}\n"
        f"核心目标：{prd.core_goals}\n"
        f"用户故事：{prd.user_stories}"
    )

    arch_res = arch_llm.invoke(prompt)
    msg_content = f"【Architect 产出设计】技术栈: {arch_res.tech_stack}\n接口: {arch_res.api_endpoints}"

    return {
        "arch_output": arch_res,
        "messages": [HumanMessage(content=msg_content, name="Architect")]
    }

# ================= 4. 构建基于 SOP 的流水线图 =================

workflow = StateGraph(MetaGPTState)

workflow.add_node("ProductManager", product_manager_role)
workflow.add_node("Architect", architect_role)

# 强约束的 SOP 顺序链路：START -> PM -> Architect -> END
workflow.add_edge(START, "ProductManager")
workflow.add_edge("ProductManager", "Architect")
workflow.add_edge("Architect", END)

app = workflow.compile()

# ================= 5. 测试执行 SOP 协作链路 =================

initial_input = {
    "user_requirement": "我想开发一个自动抓取 GitHub 热门项目并用 AI 每周生成中文简报的系统。",
    "prd_output": None,
    "arch_output": None,
    "messages": []
}

print("🚀 启动 SOP 驱动的软件开发多智能体流水线...\n")
output = app.invoke(initial_input)

print("\n--- 最终 SOP 结构化交付物 ---")
print("📄 标准 PRD 产出:")
print(output["prd_output"].model_dump_json(indent=2))
print("\n📐 标准架构设计产出:")
print(output["arch_output"].model_dump_json(indent=2))


**核心工程细节解析**
1. 使用结构化数据规范接口：通过 Pydantic 模型（如 `PRDDocument` 和 `ArchitectureDesign`），强制要求各个角色交付明确的数据对象，彻底避免了自然语言交互中的语义歧义与格式混乱。
2. SOP 流程硬约束：将软件开发的标准 SOP 硬编码（Hard-wire）在有向图的工作流中。相比于让 AI 自由选择“下一步干什么”，SOP 链路能够大幅提升任务交付的确定性与可靠性。

**思考与练习题：**
1. SOP 与自由对话的平衡：在实际工程中，什么时候应该采用 MetaGPT 式的刚性 SOP 流水线，什么时候应该采用 AutoGen 式的自由对等网络（P2P）？
2. 扩展流水线：如果需要在 `Architect` 节点之后再增加一个 `Engineer`（工程师）节点，要求其根据架构设计自动生成 `main.py` 的可执行代码，你将如何为 `Engineer` 节点定义它的 Pydantic 输入与输出结构？

## Multi-Agent 终极复盘与工程防线
Multi-Agent 落地时最核心的工程痛点——Token 爆炸与递归死循环防御。

1. 多智能体架构选型复盘：彻底弄懂在不同业务场景下，如何精准选择 Supervisor、P2P、AutoGen 或 MetaGPT 架构。
2. 攻克生产级工程痛点：掌握 Token 级成本控制、消息历史裁剪（Context Truncation）以及无限递归死循环的物理熔断机制。


**核心知识体系全景复盘**

| 架构 / 框架           | 核心拓扑机制                             | 优势                      | 劣势 / 痛点                                 | 最佳落地场景                                                      |
|-------------------|------------------------------------|-------------------------|-----------------------------------------|-------------------------------------------------------------|
| Supervisor 主控模式   | 星型（Star）：由中央 Agent 集中调度分配工作。       | 控场能力强、任务分发行明确、逻辑可追溯。    | 中央节点可能会产生单点 Bottleneck，增加一轮调度 Token 开销。 | 业务路由分流、多任务独立并行处理（如综合客服系统）。                                  |
| Peer-to-Peer 网状模式 | 网状（Mesh）：Agent 之间手递手直接传球交互。        | 链路紧凑、非常适合角色间的深度质检与辩论。   | 缺少中央控场，路径不容易预测，容易陷入死循环。                 | 交叉质检（如 Coder $\leftrightarrow$ Reviewer）、角色辩论。              |
| AutoGen (微软)      | 事件驱动（Event-driven）/ Code Execution | 自带强大的代码沙箱执行与报错自动纠错能力。   | 自由度过高，如果提示词不够收敛，容易偏离主题。                 | 数据分析自动化、代码生成与调试、科学计算。                                       |
| MetaGPT           | 刚性 SOP 流水线 / 结构化输出                 | 彻底消除无用对话，输出质量极度稳定且符合规范。 | 缺乏灵活应变能力，不适合规则未确定的探索性任务。                | 软件工程自动化（PRD $\rightarrow$ Architecture $\rightarrow$ Code）。 |

#### 生产级 Multi-Agent 的三大工程防护墙
在将多智能体系统推向生产环境（Production）时，如果不加限制，往往会因为智能体之间的“无休止扯皮”导致 Token 飙升或系统崩溃。必须构建以下三道防线：
1. 递归深度与执行时间上限,在图引擎层设定绝对的硬限制，绝不允许 Agent 无限循环。
    * 在 LangGraph 中：在 `invoke` 或 `stream` 时传入 `config={"recursion_limit": 15}`，超过 15 步强制中断并抛出异常。
    * 在 AutoGen 中：设定 `max_consecutive_auto_reply=10`。

2. 上下文记忆裁剪,多智能体系统的对话历史增长速度是单智能体的 $N$ 倍。
    * 策略 A（滑动窗口）：在节点向大模型发请求前，只截取 `messages` 列表中最新的 $K$ 条消息。
    * 策略 B（结构化传递）：像 MetaGPT 一样，Worker Agent 不读取前人的全量对话历史，只读取上游 Agent 产出的结构化 JSON/Pydantic 摘要。

3. 结构化枚举路由
    * 避免使用纯自然语言让 Supervisor 决定下一步去哪。必须强约束其输出为 Pydantic 枚举对象（例如 `Literal["Coder", "Reviewer", "FINISH"]`），防止模型因为拼写错误产生非法的节点跳转。

#### 深度工程考题
1. 考核题 1（架构选型）：
    * 你的团队准备开发一个“全自动智能量化交易助手”，需要完成：① 抓取财报数据；② 撰写量化策略 Python 代码；③ 在历史数据上回测代码并输出收益率曲线；④ 根据回测结果修改策略直至收益率达标。
    * 请问：你会选择 AutoGen、MetaGPT 还是 LangGraph 的 Supervisor 模式？请给出你的架构选型理由及核心数据流设计。

2. 考核题 2（死循环与防逃逸）：
    * 在一个由 Coder 和 Reviewer 组成的 P2P 纠错网络中，如果 Reviewer 要求极度苛刻（例如一直对变量命名不满意，导致 Coder 修改后仍无法包含 `APPROVED` 关键字），系统达到了 `recursion_limit` 物理上限而被强制中断。
    * 作为架构师，你会在 LangGraph 中设计怎样的兜底机制（Fallback Strategy），来优雅地向终端用户解释情况并输出当前“最好的一版代码”？

3. 考核题 3（上下文膨胀与成本控制）：
    * 假设 Supervisor 下有 5 个专业的 Worker Agent。随着轮次的增加，全局 `messages` 列表已经累计到了 80 条消息（约 30k Tokens）。此时 Supervisor 每次做出分配决策都要读取这 80 条消息，消耗巨大的 Token 费用。
    * 请设计一个优化方案，在不丢失此前关键结论的前提下，将 Supervisor 每次决策的上下文 Token 占用降低 80% 以上。
